<a href="https://colab.research.google.com/github/yt6363/.github-workflows-/blob/main/Copy_of_Price_Projections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Nifty Swing & Cycle Analyzer App with Projection Feature (Fixed Plot Rendering for Colab)
!pip install plotly ipywidgets --quiet

import pandas as pd
import numpy as np
import bisect
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

pio.renderers.default = 'colab'

swing_df = None
raw_df = None

# ------------------------ CORE FUNCTIONS --------------------------
def generate_cycle_levels(start=2.0, max_price=30000):
    levels = [start]
    current = start
    while current < max_price:
        current = (np.sqrt(current) + 0.25) ** 2
        levels.append(current)
    return levels

def get_cycle_fraction(price, levels):
    if price <= levels[0]: return 0.0
    if price >= levels[-1]: return (len(levels) - 1) / 8.0
    idx = bisect.bisect_right(levels, price)
    lower = levels[idx - 1]
    upper = levels[idx]
    interval_fraction = (price - lower) / (upper - lower)
    return (idx - 1 + interval_fraction) / 8.0

def get_cycle_position_string(cycle_fraction):
    full_cycles = int(cycle_fraction)
    degrees = round((cycle_fraction - full_cycles) * 360, 1)
    return f"{full_cycles} & {degrees}°"

def get_price_from_cycle(target_cycle_fraction, levels):
    idx = int(target_cycle_fraction * 8)
    if idx >= len(levels):
        return None
    return levels[idx]

# --------------------- SWING DETECTION -------------------------
def detect_swings(dates, highs=None, lows=None, closes=None, threshold=0.05, mode="HighLow"):
    dates = np.array(dates)
    n = len(dates)
    swings = []

    if mode == "HighLow":
        highs = np.array(highs, dtype=float)
        lows = np.array(lows, dtype=float)
        i = 0
        extreme_price = lows[i]
        extreme_index = i
        trend = 'looking_for_low'
        for k in range(i + 1, n):
            high = highs[k]
            low = lows[k]
            if trend == 'looking_for_top':
                if high > extreme_price:
                    extreme_price = high
                    extreme_index = k
                elif low < extreme_price * (1 - threshold):
                    swings.append((dates[extreme_index], extreme_price, "Top"))
                    trend = 'looking_for_low'
                    extreme_price = low
                    extreme_index = k
            else:
                if low < extreme_price:
                    extreme_price = low
                    extreme_index = k
                elif high > extreme_price * (1 + threshold):
                    swings.append((dates[extreme_index], extreme_price, "Low"))
                    trend = 'looking_for_top'
                    extreme_price = high
                    extreme_index = k
    else:
        prices = np.array(closes, dtype=float)
        i = 0
        extreme_price = prices[i]
        extreme_index = i
        trend = 'looking_for_low'
        for k in range(i + 1, n):
            price = prices[k]
            if trend == 'looking_for_top':
                if price > extreme_price:
                    extreme_price = price
                    extreme_index = k
                elif price < extreme_price * (1 - threshold):
                    swings.append((dates[extreme_index], extreme_price, "Top"))
                    trend = 'looking_for_low'
                    extreme_price = price
                    extreme_index = k
            else:
                if price < extreme_price:
                    extreme_price = price
                    extreme_index = k
                elif price > extreme_price * (1 + threshold):
                    swings.append((dates[extreme_index], extreme_price, "Low"))
                    trend = 'looking_for_top'
                    extreme_price = price
                    extreme_index = k

    return pd.DataFrame(swings, columns=["Date", "Price", "Type"])

# --------------------- PLOTTING -------------------------
def plot_with_projection(df, swings_df, projected=None):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['Date'], y=df['High'], mode='lines', name='High', line=dict(color='lightblue')))

    fig.add_trace(go.Scatter(
        x=swings_df['Date'], y=swings_df['Price'], mode='markers+text',
        marker=dict(color=np.where(swings_df['Type']=='Top', 'red', 'green'), size=10,
                    symbol=np.where(swings_df['Type']=='Top', 'triangle-up', 'triangle-down')),
        text=swings_df['CyclePosition'], textposition='top center', name='Swings'))

    if projected:
        fig.add_trace(go.Scatter(
            x=[projected['Date']], y=[projected['Price']], mode='markers+text',
            marker=dict(color='blue', size=12, symbol='star'),
            text=[f"Projected\n{projected['CyclePosition']}"], textposition='top center',
            name='Projection'))

    fig.update_layout(title="Swing Chart with Cycle Projections", height=600)
    fig.show()

# --------------------- ANALYSIS -------------------------
def run_analysis(file, threshold, swing_mode, timeframe):
    df = pd.read_csv(file, parse_dates=["Date"])
    df.columns = [c.strip() for c in df.columns]

    for col in ["Open", "High", "Low", "Close"]:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False)
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df.dropna(subset=["High", "Low", "Close"], inplace=True)

    if timeframe == "Weekly":
        df = df.resample('W-MON', on='Date').agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'
        }).dropna().reset_index()

    swings = detect_swings(df["Date"], df["High"], df["Low"], df["Close"], threshold, mode=swing_mode)
    swings.sort_values("Date", inplace=True)
    swings.reset_index(drop=True, inplace=True)

    levels = generate_cycle_levels(max_price=50000)
    swings["CycleFraction"] = swings["Price"].apply(lambda p: get_cycle_fraction(p, levels))
    swings["CyclePosition"] = swings["CycleFraction"].apply(get_cycle_position_string)

    return df, swings

# --------------------- PROJECTION UI ---------------------
def build_projection_ui(df, swings):
    swings['Label'] = swings.apply(lambda row: f"{row['Type']} | {row['Date'].date()} | {row['CyclePosition']}", axis=1)
    s1 = widgets.Dropdown(options=swings['Label'], description='Select 1st:')
    s2 = widgets.Dropdown(options=swings['Label'], description='Select 2nd:')
    btn = widgets.Button(description='Project Target')

    def on_click(b):
        clear_output()
        idx1 = s1.index
        idx2 = s2.index
        row1 = swings.iloc[idx1]
        row2 = swings.iloc[idx2]

        c1 = int(row1['CycleFraction'])
        c2 = int(row2['CycleFraction'])
        direction = c1 - c2
        target_cycle = c1 + direction
        target_type = row1['Type']

        levels = generate_cycle_levels(max_price=50000)
        target_price = get_price_from_cycle(target_cycle, levels)
        if target_price is None:
            print("Target cycle exceeds range.")
            return

        projected = {
            'Date': df['Date'].max(),
            'Price': target_price,
            'CyclePosition': f"{target_cycle} & 0.0°"
        }
        plot_with_projection(df, swings, projected)
        print(f"🔵 Projected {target_type} at Cycle {target_cycle}: Price ≈ {round(target_price, 2)}")

    btn.on_click(on_click)
    display(widgets.VBox([s1, s2, btn]))

# --------------------- UI CONTROLS ---------------------
upload_btn = widgets.FileUpload(accept=".csv", multiple=False)
thresh_dd = widgets.Dropdown(options=["3%", "5%", "Custom"], value="5%", description="Threshold")
thresh_val = widgets.FloatText(value=0.04, description="Custom %")
thresh_val.layout.display = 'none'
timeframe_dd = widgets.Dropdown(options=["Daily", "Weekly"], value="Daily", description="Timeframe")
swingmode_dd = widgets.Dropdown(options=["HighLow", "Close"], value="HighLow", description="Swing Type")
run_btn = widgets.Button(description="Run Analysis", button_style="success")
out_box = widgets.Output()

def on_thresh_change(change):
    thresh_val.layout.display = 'block' if change['new'] == 'Custom' else 'none'
thresh_dd.observe(on_thresh_change, names='value')

def on_run_click(b):
    clear_output()
    if upload_btn.value:
        uploaded = list(upload_btn.value.values())[0]
        fname = uploaded['metadata']['name']
        with open(fname, 'wb') as f:
            f.write(uploaded['content'])
        threshold = {"3%": 0.03, "5%": 0.05}.get(thresh_dd.value, thresh_val.value)
        df, swings = run_analysis(fname, threshold, swingmode_dd.value, timeframe_dd.value)
        global raw_df, swing_df
        raw_df, swing_df = df, swings
        plot_with_projection(df, swings)
        build_projection_ui(df, swings)
    else:
        print("⚠️ Upload a file first.")

run_btn.on_click(on_run_click)

# DISPLAY UI
display(widgets.VBox([
    widgets.HTML("<h3>📊 Nifty Swing & Cycle Analyzer + Projection Tool</h3>"),
    upload_btn,
    widgets.HBox([thresh_dd, thresh_val]),
    swingmode_dd, timeframe_dd,
    run_btn
]))
